# Reproduction: Saini et al. (2023) hybrid RF+XGBoost

Paper's methods only: 5 baselines (DT, LR, RF, XGBoost, MLP) plus the paper's RF+XGBoost averaging hybrid, with SHAP.

Set `DATASET` below to one of: `nsl_kdd`, `unsw_nb15`, `cic_ids2017`, `cse_cic_ids2018`.

Setup on Kaggle: turn Internet ON (Settings), add the saved `kaggle_prep` notebook output as an input (or the raw-data dataset, which is slower), run all cells. The code is cloned from GitHub.

In [ ]:
DATASET = "nsl_kdd"   # <-- change this line only: nsl_kdd, unsw_nb15, cic_ids2017, cse_cic_ids2018

In [ ]:
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
os.environ["RIS_DATA_PROCESSED"] = f"{WORK}/data/processed"
os.environ["RIS_RESULTS"] = f"{WORK}/results"

# Clone (or update) the repo; needs Internet turned on in notebook settings
REPO = f"{WORK}/RIS-M26"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Sahoo-Achyutananda/RIS-M26.git", REPO], check=True)
SRC = f"{REPO}/src"

# The uploaded raw folder (containing nsl_kdd/, unsw_nb15/, cic-ids-2017/, cds-ids-2018/) is read
# in place from /kaggle/input: nothing is copied, so saved versions stay small.
RAW_MARKERS = {
    "nsl_kdd": "nsl_kdd/KDDTrain+*",
    "unsw_nb15": "unsw_nb15/UNSW_NB15_training-set.csv",
    "cic_ids2017": "cic*2017/*WorkingHours*.csv",
    "cse_cic_ids2018": "c*2018/*-2018.csv",
}

def point_at_raw(datasets):
    roots, missing = set(), []
    for ds in datasets:
        hits = glob.glob(f"/kaggle/input/**/{RAW_MARKERS[ds]}", recursive=True)
        if hits:
            roots |= {os.path.dirname(os.path.dirname(h)) for h in hits}
        else:
            missing.append(ds)
    if missing or len(roots) != 1:
        for root, _, files in os.walk("/kaggle/input"):
            for fn in files:
                print("  ", os.path.join(root, fn))
        raise AssertionError(f"raw data not found for {missing}, or found in several places: {roots}")
    os.environ["RIS_DATA_RAW"] = roots.pop()
    print("raw data folder:", os.environ["RIS_DATA_RAW"])

# Prefer prepared data (output of kaggle_prep) when it is attached as an input
prepared = glob.glob(f"/kaggle/input/**/{DATASET}/dataset_final.csv", recursive=True)
SKIP_PREP = bool(prepared)
if SKIP_PREP:
    pdest = f"{WORK}/data/processed/{DATASET}"
    os.makedirs(pdest, exist_ok=True)
    shutil.copy(prepared[0], pdest)  # small file; the pipeline writes next to it
    print("using prepared data:", prepared[0])
else:
    point_at_raw([DATASET])

In [ ]:
!pip install -q imbalanced-learn shap xgboost

In [ ]:
proc = subprocess.run(
    [sys.executable, "-u", "run_pipeline.py", "--dataset", DATASET, "--mode", "reproduce"] + (["--skip-prep"] if SKIP_PREP else []),
    cwd=SRC, env=os.environ, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(proc.stdout[-6000:])
print("exit code:", proc.returncode)

In [ ]:
import pandas as pd
pd.read_csv(f"{WORK}/results/{DATASET}/metrics_reproduce.csv", index_col=0).round(4)